<a href="https://colab.research.google.com/github/TamirPalay/DI_Exercises/blob/main/week15/day1-2/Copy_of_Evaluating_LLMs_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [2]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [3]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    # Yield slices of items of length batch_size
    sliced_items = [items[i:i+batch_size] for i in range(0, len(items), batch_size)]
    for batch in sliced_items:
        yield batch

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    # Load tokenizer/model, send to device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    all_summaries = []
    for i, batch in enumerate(batch_generator(texts, batch_size)):
        # Tokenize with prefix
        prefixed_batch = [f"summarize: {text}" for text in batch]
        inputs = tokenizer(prefixed_batch, return_tensors='pt', padding=True, truncation=True).to(device)

        # Generate summaries
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

        # Decode summaries
        decoded_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_summaries.extend(decoded_summaries)

        # Clear caches between batches
        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = False
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

Skipping T5 generation for speed. Set RUN_T5=True to execute.



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [4]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Accuracy skipped (no predictions).



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [5]:

import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # TODO: normalize preds/refs; call rouge.compute
    score = rouge.compute(predictions=preds, references=refs)
    return score
    # raise NotImplementedError("Implement compute_rouge_score")

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))


ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.


### Findings from ROUGE sanity check

The ROUGE scores were calculated for a set of test predictions and references. The results show that for exact matches, ROUGE scores are high (e.g., 0.66 for the given example), while predictions that are empty or do not match the reference will yield lower scores, as expected. This confirms the basic functionality of the `compute_rouge_score` function.


### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [ ]:
import torch, gc
import pandas as pd # Ensure pandas is imported for DataFrame operations
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List

# Assuming batch_generator and normalize_text from previous cells are available

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # GPT-2 does not have a pad_token by default, which is needed for batch generation.
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    all_summaries = []
    for i, batch in enumerate(batch_generator(texts, batch_size)):
        prefixed_batch = [f"TL;DR: {text}" for text in batch]
        # Added max_length for truncation as GPT-2 can handle longer inputs
        inputs = tokenizer(prefixed_batch, return_tensors='pt', padding=True, truncation=True, max_length=1024).to(device)

        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode summaries, slicing to remove the input prompt
        decoded_summaries = [tokenizer.decode(output[len(inputs.input_ids[j]):], skip_special_tokens=True) for j, output in enumerate(outputs)]
        all_summaries.extend(decoded_summaries)

        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    return all_summaries


def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    rouge_1_scores = []
    rouge_2_scores = []
    rouge_l_scores = []
    for index, row in df.iterrows():
        pred = row[pred_col]
        ref = row[ref_col]
        # Ensure preds and refs are strings, handle potential non-string values
        pred_str = str(pred) if pd.notna(pred) else ""
        ref_str = str(ref) if pd.notna(ref) else ""

        # Normalize text using the helper function from cell 3ef0a7df
        normalized_pred = normalize_text(pred_str)
        normalized_ref = normalize_text(ref_str)

        # rouge.compute expects lists of strings
        score = rouge.compute(predictions=[normalized_pred], references=[normalized_ref])
        rouge_1_scores.append(score['rouge1'])
        rouge_2_scores.append(score['rouge2'])
        rouge_l_scores.append(score['rougeL'])

    df[f'rouge1_{pred_col}'] = rouge_1_scores
    df[f'rouge2_{pred_col}'] = rouge_2_scores
    df[f'rougeL_{pred_col}'] = rouge_l_scores
    return df

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_COMPARE = True # Setting to True to run the comparison
if RUN_COMPARE:
    print("Running model comparisons. This may take a while...")

    # Generate all summaries
    print("Generating t5-small summaries...")
    # The previous cell 7111176d only defined train_summaries_t5 if RUN_T5 was True.
    # We ensure it's generated here if RUN_COMPARE is True.
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2, max_new_tokens=32)

    print("Generating t5-base summaries...")
    train_summaries_t5_base = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-base', batch_size=2, max_new_tokens=32)

    print("Generating gpt2 summaries...")
    train_summaries_gpt2 = summarize_with_gpt2(train_df['prompt_text'].tolist(), model_name='gpt2', batch_size=2, max_new_tokens=32)

    comparison_df = pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5,
        't5_base_summary': train_summaries_t5_base,
        'gpt2_summary': train_summaries_gpt2
    })

    print("Computing ROUGE scores for t5-small...")
    comparison_df = compute_rouge_per_row(comparison_df, 't5_small_summary')
    print("Computing ROUGE scores for t5-base...")
    comparison_df = compute_rouge_per_row(comparison_df, 't5_base_summary')
    print("Computing ROUGE scores for gpt2...")
    comparison_df = compute_rouge_per_row(comparison_df, 'gpt2_summary')

    display(comparison_df.head())
else:
    print("Skipping model comparison. Set RUN_COMPARE=True to execute.")

Running model comparisons. This may take a while...
Generating t5-small summaries...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [ ]:
import pandas as pd
from typing import List

def compare_models(comparison_df: pd.DataFrame, model_summary_cols: List[str]):
    """
    Calculates the average ROUGE-1, ROUGE-2, and ROUGE-L scores for each model.

    Args:
        comparison_df: DataFrame containing per-row ROUGE scores (e.g., 'rouge1_t5_small_summary').
        model_summary_cols: List of column names in comparison_df that contain the model summaries
                            (e.g., ['t5_small_summary', 't5_base_summary', 'gpt2_summary']).
    Returns:
        A DataFrame with average ROUGE scores for each model.
    """
    results = {}
    for col in model_summary_cols:
        model_name = col.replace('_summary', '') # Extract model name from column
        rouge1_col = f'rouge1_{col}'
        rouge2_col = f'rouge2_{col}'
        rougeL_col = f'rougeL_{col}'

        if rouge1_col in comparison_df.columns:
            results[model_name] = {
                'rouge1': comparison_df[rouge1_col].mean(),
                'rouge2': comparison_df[rouge2_col].mean(),
                'rougeL': comparison_df[rougeL_col].mean()
            }
    return pd.DataFrame.from_dict(results, orient='index')


def compare_models_summaries(df: pd.DataFrame, pred_cols: List[str]):
    """
    Subsets columns for side-by-side viewing of model summaries.

    Args:
        df: The DataFrame containing original texts, reference, and model summaries.
        pred_cols: A list of column names for the model predictions (e.g., 't5_small_summary').

    Returns:
        A DataFrame with selected columns for comparison.
    """
    display_cols = ['prompt_text', 'reference_summary'] + pred_cols
    return df[display_cols]


## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.
